# 22. Multimodal RAG with Captioning

Caption images with a vision LLM, then index captions next to the text so one text index serves both.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Charts, diagrams and photos contain facts that text-only RAG cannot see. The simplest multimodal approach is **captioning**: a vision-language model turns each image into a detailed text description, and those captions are indexed next to the normal text chunks. Everything downstream (embedding, search, generation) stays text-only, and results can point back to the original image.

## How it works

1. Extract or generate the images
2. Caption each with a vision LLM (qwen2.5vl / llava / moondream)
3. Embed the captions
4. Store them in the same index as text, with `type=image` metadata and the file path
5. Retrieve; show the image when an image-caption is hit

```text
 image --> vision LLM --> caption text --\
                                          +--> one text index --> retrieve --> LLM (+ show image)
 text chunks ---------------------------/
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** matplotlib, `ollama pull qwen2.5vl:3b`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import ollama

figures = make_figures()          # renders 3 sample chart PNGs into data/figures
print([f.name for f in figures])

## Caption every image
A vision model turns each chart into searchable text.

In [ ]:
def caption(path):
    reply = ollama.chat(model=VISION_MODEL, messages=[{
        "role": "user",
        "content": "Describe this chart in detail: title, axes, every value and the main takeaway.",
        "images": [str(path)],
    }])
    return reply["message"]["content"].strip()

captions = {f.name: caption(f) for f in figures}
for name, cap in captions.items():
    print(name, "->", cap[:200], "\n")

## One index for text and images
Caption chunks keep a pointer to the source image so you can show it with the answer.

In [ ]:
mm_store = VectorStore().add(chunks, [{"type": "text"} for _ in chunks])
mm_store.add(list(captions.values()), [{"type": "image", "file": n} for n in captions])

query = "Which retrieval method achieves the best recall?"
hits = mm_store.search(query, 3)
for h in hits: print(h["meta"], f"{h['score']:.3f}")
print(answer(query, [h["text"] for h in hits]))

In [ ]:
from IPython.display import Image, display
for h in hits:
    if h["meta"]["type"] == "image":
        display(Image(filename=str(DATA / "figures" / h["meta"]["file"])))
        break

## Show the images the model captioned

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(figures), figsize=(13, 3.5))
for ax, f in zip(axes, figures):
    ax.imshow(plt.imread(f)); ax.set_title(f.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## Caption quality check
Compare what the model said with what is actually in the chart (the values are known: we generated them).

In [ ]:
truth = {"chunk_size_vs_hit_rate.png": ["200", "400", "800", "1600"],
         "latency_breakdown.png": ["Embed", "Search", "Rerank", "Generate"],
         "retrieval_methods_recall.png": ["Dense", "BM25", "Hybrid", "Rerank"]}
for name, cap in captions.items():
    found = [t for t in truth[name] if t.lower() in cap.lower()]
    print(f"{name}: mentions {len(found)}/{len(truth[name])} expected labels -> {found}")

## Cross-modal questions
Questions answerable only from the images should retrieve image captions; text questions should still hit text.

In [ ]:
for q in ["Which pipeline stage takes the most time?", "Which chunk size has the highest hit rate in the chart?", "What does a cross-encoder do?"]:
    hits = mm_store.search(q, 3)
    print(f"Q: {q}\n   types retrieved: {[h['meta']['type'] for h in hits]}")
    print("   A:", answer(q, [h["text"] for h in hits])[:200], "\n")

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Reuses your whole text RAG stack | Caption quality bounds retrieval quality |
| Easy to cite the original image | Loses visual details the caption omits |
| Small local vision models work | One vision-LLM call per image |

## When to use it

Documents with charts and screenshots where exact visuals matter less than their content. For layout-heavy pages, see ColPali (23).

## Experiments to try

- Ask the vision model for structured output (title, axes, values).
- Add OCR text to the caption.
- Compare `qwen2.5vl:3b` and `moondream` captions.

## Key takeaways

- Convert images to text once, index like any text.
- Keep a pointer to the original image.
- Always spot-check captions.

---
**Next:** [23. Multimodal RAG with ColPali](../23_multi_model_rag_with_colpali/multi_model_rag_with_colpali.ipynb)